# US Accidents — Random Forest Training (PySpark + GPU via cuML)

## ⚠️ Important GPU Note

**PySpark MLlib's `RandomForestClassifier` has NO native GPU support.**  
This is a fundamental difference from XGBoost, which exposes `device='cuda'`.

| Path | Library | GPU? | Scale |
|------|---------|------|-------|
| Primary | `pyspark.ml.classification.RandomForestClassifier` | ❌ CPU only | Distributed |
| GPU alternative | `cuml.ensemble.RandomForestClassifier` (RAPIDS) | ✅ CUDA | Single-node GPU |

**Strategy in this notebook:**
- Always train the full distributed model with PySpark MLlib (CPU, handles any dataset size).
- If a GPU + RAPIDS is detected, also train a cuML GPU model on a representative sample and compare metrics side-by-side.

**Pipeline:**
1. Environment & GPU/cuML detection
2. SparkSession setup
3. Load feature-engineered train / test splits
4. Label alignment & class-weight column
5. Baseline RandomForest (PySpark MLlib, CPU)
6. Hyperparameter tuning — `CrossValidator`
7. Full evaluation — accuracy, weighted-F1, per-class report, confusion matrix
8. Feature importance (MDI)
9. OOB error analysis
10. GPU path — cuML RandomForest (if RAPIDS available)
11. Model comparison table
12. Save model + artifacts

**Dependencies:**
```bash
pip install pyspark matplotlib seaborn scikit-learn
# GPU path (optional — requires CUDA 11.x + NVIDIA driver):
pip install cuml-cu11   # or cuml-cu12 for CUDA 12.x
```

In [1]:
# ── 0. Env vars — BEFORE any pyspark import ──────────────────────────────────
import os

HADOOP_HOME = os.getenv("HADOOP_HOME", r"C:\hadoop")
os.environ["HADOOP_HOME"]            = HADOOP_HOME
os.environ["PATH"]                  += os.pathsep + os.path.join(HADOOP_HOME, "bin")
os.environ["PYSPARK_PYTHON"]         = "python"
os.environ["PYSPARK_DRIVER_PYTHON"]  = "python"

In [15]:
# ── 1. GPU & cuML Detection ───────────────────────────────────────────────────
import subprocess

def detect_gpu() -> dict:
    """Probe nvidia-smi for available CUDA devices."""
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        rows = [r.split(",") for r in raw.splitlines() if r]
        gpus = [{"name": r[0].strip(), "vram": r[1].strip(), "driver": r[2].strip()}
                for r in rows]
        return {"available": True, "count": len(gpus), "gpus": gpus}
    except Exception:
        return {"available": False, "count": 0, "gpus": []}


def detect_cuml() -> bool:
    """Check whether RAPIDS cuML is importable."""
    try:
        import cuml  # noqa: F401
        return True
    except ImportError:
        return False


GPU_INFO = detect_gpu()
HAS_CUML = detect_cuml() and GPU_INFO["available"]
USE_GPU  = HAS_CUML

print("=" * 60)
print("  HARDWARE DETECTION")
print("=" * 60)
if GPU_INFO["available"]:
    for g in GPU_INFO["gpus"]:
        print(f"  GPU  : {g['name']}  |  VRAM: {g['vram']}  |  Driver: {g['driver']}")
else:
    print("  GPU  : Not detected")

print(f"  cuML : {'✅ available — GPU path enabled' if HAS_CUML else '❌ not found — CPU path only'}")
print()
print("  ⚠️  PySpark MLlib RandomForest = CPU always (by design)")
print("      GPU acceleration via RAPIDS cuML on a sampled dataset.")
print("=" * 60)

  HARDWARE DETECTION
  GPU  : Quadro M1000M  |  VRAM: 4096 MiB  |  Driver: 582.41
  cuML : ❌ not found — CPU path only

  ⚠️  PySpark MLlib RandomForest = CPU always (by design)
      GPU acceleration via RAPIDS cuML on a sampled dataset.


In [16]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import json
import time
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.ticker    as mticker
import matplotlib.gridspec  as gridspec
import seaborn              as sns

warnings.filterwarnings("ignore")

# PySpark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.ml                      import Pipeline
from pyspark.ml.classification       import RandomForestClassifier
from pyspark.ml.evaluation           import MulticlassClassificationEvaluator
from pyspark.ml.tuning               import CrossValidator, ParamGridBuilder

# scikit-learn (driver-side evaluation on Pandas)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("All imports OK")

All imports OK


In [17]:
# ── 3. SparkSession ───────────────────────────────────────────────────────────
# RandomForest in MLlib is memory-intensive for large forests.
# Increase driver + executor memory if you hit OOM with many trees.

spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

Spark version : 4.1.1
Spark UI      : http://192.168.1.19:4040


In [18]:
# ── 4. Load Feature-Engineered Data ──────────────────────────────────────────
DATA_BASE = "Data/Feature_Engineered_Data"

In [19]:
train_df = spark.read.parquet(f"{DATA_BASE}/train")
test_df  = spark.read.parquet(f"{DATA_BASE}/test")

In [20]:
n_train = train_df.count()
n_test  = test_df.count()

In [21]:
print(f"Train  rows : {n_train:>12,}")
print(f"Test   rows : {n_test:>12,}")
print(f"Columns     : {len(train_df.columns)}")

Train  rows :    8,458,433
Test   rows :    2,115,303
Columns     : 68


In [22]:
train_df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- ID: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 

In [23]:
# ── 5. Label Engineering & Constants ─────────────────────────────────────────
# Spark MLlib RandomForestClassifier requires:
#   labelCol    : DoubleType, zero-indexed  (0.0 … n_classes-1)
#   featuresCol : Vector
#   weightCol   : DoubleType  (computed in feature engineering notebook)

LABEL_COL   = "label"
FEATURE_COL = "features"    # change to 'scaled_features' if you scaled
WEIGHT_COL  = "weight"
NUM_CLASSES = 4
SEED        = 42

CLASS_NAMES = [f"Severity {i + 1}" for i in range(NUM_CLASSES)]

# Shift Severity 1–4  →  label 0–3
train_df = train_df.withColumn(LABEL_COL, (col("Severity") - 1).cast("double"))
test_df  = test_df .withColumn(LABEL_COL, (col("Severity") - 1).cast("double"))

print("Train class distribution:")
train_df.groupBy("Severity", LABEL_COL) \
        .count() \
        .orderBy("Severity") \
        .show()

print("Class weights:")
train_df.groupBy("Severity") \
        .agg(F.first(WEIGHT_COL).alias("weight")) \
        .orderBy("Severity") \
        .show()

Train class distribution:
+--------+-----+-------+
|Severity|label|  count|
+--------+-----+-------+
|       1|  0.0|  74893|
|       2|  1.0|6950921|
|       3|  2.0|1163736|
|       4|  3.0| 268883|
+--------+-----+-------+

Class weights:
+--------+-------------------+
|Severity|             weight|
+--------+-------------------+
|       1| 28.296534966120383|
|       2|0.30419361535342115|
|       3|  1.817490143986491|
|       4|  7.869634983909045|
+--------+-------------------+



In [24]:
# ── 6. RandomForest Model Definition ─────────────────────────────────────────
# Key parameter choices explained:
#
#  numTrees          — More trees = lower variance, higher cost.
#                      200–500 is typically the sweet spot for tabular data.
#  maxDepth          — Deeper trees overfit; RF mitigates this via bagging.
#                      Start at 10–15; tune up or down.
#  maxBins           — Must be ≥ max unique values of any categorical feature.
#                      High-cardinality StringIndexer cols need a high value.
#  featureSubsetStrategy
#                    — 'sqrt'  : classic RF (sqrt(n_features) per split)
#                      'log2'  : aggressive reduction, faster, sometimes better
#                      'auto'  : 'sqrt' for classification
#  impurity          — 'gini' (default) or 'entropy'; gini is faster
#  subsamplingRate   — Fraction of rows per tree (bagging fraction).
#                      0.8 reduces variance without too much bias.
#  minInstancesPerNode — Minimum leaf size; prevents over-specific leaves.

rf = RandomForestClassifier(
    labelCol             = LABEL_COL,
    featuresCol          = FEATURE_COL,
    weightCol            = WEIGHT_COL,
    predictionCol        = "prediction",
    probabilityCol       = "probability",
    rawPredictionCol     = "rawPrediction",
    numTrees             = 300,
    maxDepth             = 12,
    maxBins              = 512,          # high — covers high-cardinality idx cols
    featureSubsetStrategy= "sqrt",
    impurity             = "gini",
    subsamplingRate      = 0.8,
    minInstancesPerNode  = 5,
    minInfoGain          = 0.0,
    seed                 = SEED,
)

print("RandomForestClassifier configured:")
print(f"  numTrees            : {rf.getNumTrees()}")
print(f"  maxDepth            : {rf.getMaxDepth()}")
print(f"  maxBins             : {rf.getMaxBins()}")
print(f"  featureSubsetStrategy: {rf.getFeatureSubsetStrategy()}")
print(f"  subsamplingRate     : {rf.getSubsamplingRate()}")
print(f"  impurity            : {rf.getImpurity()}")
print(f"  minInstancesPerNode : {rf.getMinInstancesPerNode()}")

RandomForestClassifier configured:
  numTrees            : 300
  maxDepth            : 12
  maxBins             : 512
  featureSubsetStrategy: sqrt
  subsamplingRate     : 0.8
  impurity            : gini
  minInstancesPerNode : 5


In [ ]:
from pyspark.ml.functions import vector_to_array

# ── 7. Baseline Training ──────────────────────────────────────────────────────
print("Training baseline RandomForest (PySpark MLlib — CPU)...")
print("Note: RF training time scales with  numTrees × maxDepth × dataset size.")
print("      Expect several minutes on large datasets.\n")

t0 = time.time()
def _drop_invalid_feature_rows(df, label):
    feature_array = vector_to_array(F.col(FEATURE_COL))
    invalid_count = F.aggregate(
        feature_array,
        F.lit(0),
        lambda acc, x: acc + F.when(
            F.isnan(x) | (x == float("inf")) | (x == float("-inf")),
            1
        ).otherwise(0)
    )

    cleaned = (
        df.withColumn("_invalid_feature_count", invalid_count)
          .filter(F.col("_invalid_feature_count") == 0)
          .drop("_invalid_feature_count")
    )

    n_before = df.count()
    n_after = cleaned.count()
    if n_after != n_before:
        print(f"Removed {n_before - n_after:,} rows with NaN/Inf in {FEATURE_COL} from {label}")
    return cleaned

train_df = _drop_invalid_feature_rows(train_df, "train")
test_df  = _drop_invalid_feature_rows(test_df,  "test")

rf_model = rf.fit(train_df)
elapsed  = time.time() - t0

print(f"Training complete in {elapsed:.1f}s  ({elapsed / 60:.1f} min)")
print(f"Number of trees : {rf_model.getNumTrees}")

Training baseline RandomForest (PySpark MLlib — CPU)...
Note: RF training time scales with  numTrees × maxDepth × dataset size.
      Expect several minutes on large datasets.

Removed 2,489,169 rows with NaN/Inf in features from train
Removed 623,228 rows with NaN/Inf in features from test


In [ ]:
# ── 8. Hyperparameter Tuning with CrossValidator ──────────────────────────────
# Grid size = 2 × 2 × 2 = 8 configs × 3 folds = 24 fits.
# Expand the grid for a more thorough search; reduce for faster iteration.

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees,  [200, 400])
    .addGrid(rf.maxDepth,  [10,  15])
    .addGrid(rf.minInstancesPerNode, [3, 10])
    .build()
)

evaluator = MulticlassClassificationEvaluator(
    labelCol      = LABEL_COL,
    predictionCol = "prediction",
    metricName    = "f1",
)

cv = CrossValidator(
    estimator          = rf,
    estimatorParamMaps = param_grid,
    evaluator          = evaluator,
    numFolds           = 3,
    seed               = SEED,
    parallelism        = 2,   # lower to 1 if driver OOMs during tuning
)

print(f"Grid : {len(param_grid)} configs  ×  3 folds  =  {len(param_grid) * 3} fits")
print("Running CrossValidator...")
t0 = time.time()

cv_model   = cv.fit(train_df)
best_model = cv_model.bestModel
elapsed    = time.time() - t0

print(f"Tuning complete in {elapsed:.1f}s  ({elapsed / 60:.1f} min)")

avg_metrics = cv_model.avgMetrics
best_idx    = int(np.argmax(avg_metrics))

print(f"\nBest config index : {best_idx}")
print(f"Best CV F1 score  : {avg_metrics[best_idx]:.4f}")
print("\nAll CV F1 scores:")
for i, (params, score) in enumerate(zip(param_grid, avg_metrics)):
    marker = "  ← best" if i == best_idx else ""
    print(
        f"  [{i:2d}] numTrees={params[rf.numTrees]:>3}  "
        f"maxDepth={params[rf.maxDepth]:>2}  "
        f"minInstances={params[rf.minInstancesPerNode]:>2}  "
        f"→ F1={score:.4f}{marker}"
    )

print(f"\nBest model params:")
print(f"  numTrees            : {best_model.getNumTrees}")
print(f"  maxDepth            : {best_model.getMaxDepth()}")
print(f"  minInstancesPerNode : {best_model.getMinInstancesPerNode()}")

In [ ]:
# ── 9. Inference on Test Set ──────────────────────────────────────────────────
predictions = best_model.transform(test_df)

predictions.select(
    LABEL_COL, "prediction", "probability"
).show(10, truncate=False)

In [ ]:
# ── 10. Spark ML Evaluation Metrics ──────────────────────────────────────────
def spark_metrics(predictions_df, label_col=LABEL_COL, pred_col="prediction"):
    """Compute all standard multi-class metrics via Spark evaluator."""
    ev = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol=pred_col
    )
    results = {}
    for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        ev.setMetricName(metric)
        results[metric] = ev.evaluate(predictions_df)
    return results


metrics = spark_metrics(predictions)

print("=" * 50)
print("  TEST SET PERFORMANCE  (best tuned model)")
print("=" * 50)
print(f"  Accuracy           : {metrics['accuracy']:.4f}")
print(f"  Weighted F1        : {metrics['f1']:.4f}")
print(f"  Weighted Precision : {metrics['weightedPrecision']:.4f}")
print(f"  Weighted Recall    : {metrics['weightedRecall']:.4f}")
print("=" * 50)

In [ ]:
# ── 11. Per-Class Report & Confusion Matrix ───────────────────────────────────
pred_pd = predictions.select(LABEL_COL, "prediction").toPandas()
y_true  = pred_pd[LABEL_COL].astype(int)
y_pred  = pred_pd["prediction"].astype(int)

print("Classification Report (test set):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ── Confusion matrix (row-normalised) ────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalised
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=CLASS_NAMES
).plot(ax=axes[0], cmap="Blues", colorbar=True, values_format=".2f")
axes[0].set_title("Confusion Matrix\n(row-normalised)",
                  fontsize=12, fontweight="bold")

# Raw counts
cm_raw = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(
    confusion_matrix=cm_raw, display_labels=CLASS_NAMES
).plot(ax=axes[1], cmap="Oranges", colorbar=True, values_format=",d")
axes[1].set_title("Confusion Matrix\n(raw counts)",
                  fontsize=12, fontweight="bold")

plt.suptitle("RandomForest — Test Set Confusion",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("rf_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → rf_confusion_matrix.png")

In [ ]:
# ── 12. Feature Importance — Mean Decrease in Impurity (MDI) ─────────────────
# RandomForestClassificationModel exposes `.featureImportances`
# as a SparseVector of MDI scores — one per input feature.

FEATURE_COLS = [
    "Start_Lat", "Start_Lng",
    "Lat_Bucket", "Lng_Bucket",
    "Distance(mi)",
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)",
    "Traffic_Signal", "Junction", "Crossing", "Stop",
    "Hour", "DayOfWeek", "Month", "IsWeekend",
    "Road_Risk_Score", "Bad_Weather_Flag", "Log_Distance",
    "Weather_Condition_idx",
    "Wind_Direction_idx",
    "Season_idx",
    "Side_idx",
    "Sunrise_Sunset_idx",
]

fi_scores = best_model.featureImportances.toArray()

# Pad or trim if vector length differs from FEATURE_COLS
n_feats   = min(len(fi_scores), len(FEATURE_COLS))
fi_df     = pd.DataFrame({
    "feature"    : FEATURE_COLS[:n_feats],
    "importance" : fi_scores[:n_feats],
}).sort_values("importance", ascending=False).reset_index(drop=True)

TOP_N = 20
top_fi = fi_df.head(TOP_N)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart — top N
palette = sns.color_palette("viridis", TOP_N)
axes[0].barh(
    top_fi["feature"][::-1],
    top_fi["importance"][::-1],
    color=palette[::-1],
)
axes[0].set_xlabel("Mean Decrease in Impurity (MDI)", fontsize=11)
axes[0].set_title(f"Top {TOP_N} Feature Importances",
                  fontsize=13, fontweight="bold")
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:.3f}")
)

# Cumulative importance
cumulative = fi_df["importance"].cumsum()
axes[1].plot(range(1, len(fi_df) + 1), cumulative,
             color="steelblue", linewidth=2)
axes[1].axhline(0.80, color="tomato",   linestyle="--", label="80% threshold")
axes[1].axhline(0.95, color="darkorange",linestyle="--", label="95% threshold")
idx_80 = int((cumulative >= 0.80).idxmax()) + 1
idx_95 = int((cumulative >= 0.95).idxmax()) + 1
axes[1].axvline(idx_80, color="tomato",    linestyle=":", alpha=0.6)
axes[1].axvline(idx_95, color="darkorange",linestyle=":", alpha=0.6)
axes[1].text(idx_80 + 0.3, 0.75, f"{idx_80} features", color="tomato",    fontsize=10)
axes[1].text(idx_95 + 0.3, 0.90, f"{idx_95} features", color="darkorange",fontsize=10)
axes[1].set_xlabel("Number of features (sorted by importance)", fontsize=11)
axes[1].set_ylabel("Cumulative importance",                     fontsize=11)
axes[1].set_title("Cumulative Feature Importance",
                  fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("RandomForest — Feature Importance (MDI)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → rf_feature_importance.png")
print()
print(fi_df.to_string(index=False))

In [ ]:
# ── 13. Per-Tree Depth Distribution ──────────────────────────────────────────
# Inspect the actual depths of the trained trees to understand whether
# the forest hit maxDepth or converged earlier (a sign of clean splits).
#
# We extract depth from each tree's debug string — quick and portable.

def tree_depth(tree) -> int:
    """Heuristic: count 'If' lines in the tree debug string as a depth proxy."""
    lines = tree.toDebugString.split("\n")
    depths = []
    for line in lines:
        stripped = line.lstrip(" |")
        if stripped.startswith("If") or stripped.startswith("Predict"):
            depth = (len(line) - len(line.lstrip(" "))) // 2
            depths.append(depth)
    return max(depths) if depths else 0


SAMPLE_TREES = min(50, best_model.getNumTrees)  # don't iterate all 400 — expensive
depths = [tree_depth(best_model.trees[i]) for i in range(SAMPLE_TREES)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(depths, bins=20, color="steelblue", edgecolor="white", linewidth=0.6)
ax.axvline(np.mean(depths), color="tomato", linestyle="--",
           label=f"Mean depth = {np.mean(depths):.1f}")
ax.set_xlabel("Tree depth",    fontsize=11)
ax.set_ylabel("Count",         fontsize=11)
ax.set_title(
    f"Depth distribution across {SAMPLE_TREES} sampled trees  "
    f"(maxDepth={best_model.getMaxDepth()})",
    fontsize=12, fontweight="bold"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("rf_tree_depth_dist.png", dpi=150)
plt.show()
print(f"Depth stats on {SAMPLE_TREES} sampled trees:")
print(f"  Min  : {min(depths)}")
print(f"  Max  : {max(depths)}")
print(f"  Mean : {np.mean(depths):.2f}")
print(f"  Std  : {np.std(depths):.2f}")
print("Saved → rf_tree_depth_dist.png")

In [ ]:
# ── 14. Probability Calibration Analysis ─────────────────────────────────────
# RandomForest probability estimates are known to be poorly calibrated
# (biased toward 0 and 1) due to averaging vote fractions.
# Plot a reliability diagram per class so you know whether to apply
# Platt scaling or isotonic regression before using probabilities downstream.

from sklearn.calibration import calibration_curve

CALIB_SAMPLE = 20_000
prob_pd = (
    predictions
    .select(LABEL_COL, "probability")
    .limit(CALIB_SAMPLE)
    .toPandas()
)

# Extract per-class probability columns
probs = np.array([v.toArray() for v in prob_pd["probability"]])
labels = prob_pd[LABEL_COL].values

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for cls in range(NUM_CLASSES):
    y_bin   = (labels == cls).astype(int)
    p_cls   = probs[:, cls]
    frac_pos, mean_pred = calibration_curve(y_bin, p_cls, n_bins=15)

    ax = axes[cls]
    ax.plot([0, 1], [0, 1], "k--",  linewidth=1.0, label="Perfect calibration")
    ax.plot(mean_pred, frac_pos,
            marker="o", markersize=5, color="steelblue", linewidth=2,
            label="Random Forest")
    ax.fill_between(mean_pred, frac_pos, mean_pred,
                    alpha=0.15, color="tomato", label="Calibration gap")
    ax.set_xlabel("Mean predicted probability", fontsize=10)
    ax.set_ylabel("Fraction of positives",      fontsize=10)
    ax.set_title(f"{CLASS_NAMES[cls]} — Reliability Diagram",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle(
    "RandomForest — Probability Calibration (Reliability Diagrams)\n"
    "Curves below the diagonal = overconfident; above = underconfident",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("rf_calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → rf_calibration.png")
print()
print("TIP: If curves deviate significantly from the diagonal, apply")
print("     sklearn.calibration.CalibratedClassifierCV (isotonic or sigmoid)")
print("     on a held-out calibration split before using probabilities.")

In [ ]:
# ── 15. GPU Path — cuML RandomForest (RAPIDS) ─────────────────────────────────
# cuML runs on a SINGLE GPU — it's not distributed across workers.
# Strategy: collect a representative sample from Spark → fit cuML on GPU.
# For datasets that exceed VRAM, reduce GPU_SAMPLE_SIZE.

cuml_metrics = None   # will be filled if GPU path runs

if HAS_CUML:
    from cuml.ensemble import RandomForestClassifier as cuRF
    import cudf

    GPU_SAMPLE_TRAIN = 200_000   # adjust to your VRAM
    GPU_SAMPLE_TEST  =  40_000

    print(f"Collecting {GPU_SAMPLE_TRAIN:,} train rows to GPU memory...")

    def spark_to_cudf(df, n_rows):
        """Sample Spark DataFrame → cuDF (GPU DataFrame)."""
        pan = (
            df.select("features", LABEL_COL)
              .limit(n_rows)
              .toPandas()
        )
        X = np.array([v.toArray() for v in pan["features"]], dtype=np.float32)
        y = pan[LABEL_COL].values.astype(np.int32)
        return cudf.DataFrame(X), cudf.Series(y)

    X_train_cu, y_train_cu = spark_to_cudf(train_df, GPU_SAMPLE_TRAIN)
    X_test_cu,  y_test_cu  = spark_to_cudf(test_df,  GPU_SAMPLE_TEST)

    print(f"cuDF train shape : {X_train_cu.shape}")
    print("Training cuML RandomForest on GPU...")

    t0 = time.time()
    cu_rf = cuRF(
        n_estimators     = 300,
        max_depth        = 12,
        n_streams        = 8,         # parallel tree-building streams on GPU
        max_features     = "sqrt",
        min_samples_leaf = 5,
        random_state     = SEED,
        verbose          = 0,
    )
    cu_rf.fit(X_train_cu, y_train_cu)
    gpu_elapsed = time.time() - t0

    print(f"cuML GPU training complete in {gpu_elapsed:.1f}s  ({gpu_elapsed/60:.1f} min)")

    y_pred_cu = cu_rf.predict(X_test_cu)

    # Pull back to CPU for sklearn metrics
    y_true_np = y_test_cu.to_numpy()
    y_pred_np = y_pred_cu.to_numpy()

    cuml_metrics = {
        "accuracy": float((y_true_np == y_pred_np).mean()),
    }

    print("\ncuML RandomForest — GPU results:")
    print(classification_report(
        y_true_np, y_pred_np,
        target_names=CLASS_NAMES,
    ))
    print(f"  GPU train time    : {gpu_elapsed:.1f}s")
    print(f"  Train sample used : {GPU_SAMPLE_TRAIN:,}")
    print(f"  Test  sample used : {GPU_SAMPLE_TEST:,}")
    print("  ⚠️  cuML results are on a SAMPLE, not the full dataset.")
    print("       Use PySpark model for production inference.")

else:
    print("cuML not available — GPU path skipped.")
    print("To enable: pip install cuml-cu11  (requires CUDA 11.x)")
    print("           pip install cuml-cu12  (requires CUDA 12.x)")

In [ ]:
# ── 16. Model Comparison Table ────────────────────────────────────────────────
comparison_rows = [
    {
        "Model"         : "RF — Baseline (PySpark)",
        "Backend"       : "CPU (distributed)",
        "Data"          : "Full dataset",
        "Accuracy"      : round(spark_metrics(rf_model.transform(test_df))["accuracy"], 4),
        "Weighted F1"   : round(spark_metrics(rf_model.transform(test_df))["f1"], 4),
    },
    {
        "Model"         : "RF — Tuned (PySpark CrossValidator)",
        "Backend"       : "CPU (distributed)",
        "Data"          : "Full dataset",
        "Accuracy"      : round(metrics["accuracy"], 4),
        "Weighted F1"   : round(metrics["f1"], 4),
    },
]

if cuml_metrics:
    comparison_rows.append({
        "Model"      : "RF — cuML (RAPIDS GPU)",
        "Backend"    : f"GPU — {GPU_INFO['gpus'][0]['name']}",
        "Data"       : f"Sample {GPU_SAMPLE_TRAIN:,} rows",
        "Accuracy"   : round(cuml_metrics["accuracy"], 4),
        "Weighted F1": "—",   # not computed above; add if needed
    })

cmp_df = pd.DataFrame(comparison_rows)
print("=" * 90)
print("  MODEL COMPARISON")
print("=" * 90)
print(cmp_df.to_string(index=False))
print("=" * 90)
print("\nNote: Use the tuned PySpark model for production — it runs")
print("      on the full dataset and integrates natively with Spark pipelines.")

In [ ]:
# ── 17. Save Model & Artifacts ────────────────────────────────────────────────
MODEL_DIR = "Models/randomforest_severity"

# 17a. PySpark MLlib model
best_model.write().overwrite().save(f"{MODEL_DIR}/spark_model")
print(f"Spark model saved    → {MODEL_DIR}/spark_model")

# 17b. cuML model (if trained)
if HAS_CUML and cuml_metrics:
    import pickle
    with open(f"{MODEL_DIR}/cuml_model.pkl", "wb") as f:
        pickle.dump(cu_rf, f)
    print(f"cuML model saved     → {MODEL_DIR}/cuml_model.pkl")

# 17c. Feature importance CSV
fi_df.to_csv(f"{MODEL_DIR}/feature_importance.csv", index=False)
print(f"Feature importance   → {MODEL_DIR}/feature_importance.csv")

# 17d. Metrics + metadata JSON
meta = {
    "model"         : "RandomForestClassifier (PySpark MLlib)",
    "num_classes"   : NUM_CLASSES,
    "backend"       : "CPU (PySpark distributed)",
    "best_num_trees": int(best_model.getNumTrees),
    "best_max_depth": int(best_model.getMaxDepth()),
    "best_min_inst" : int(best_model.getMinInstancesPerNode()),
    "best_cv_f1"    : round(float(cv_model.avgMetrics[best_idx]), 4),
    "test_metrics"  : {k: round(v, 4) for k, v in metrics.items()},
    "feature_cols"  : FEATURE_COLS,
    "gpu_available" : GPU_INFO["available"],
    "cuml_trained"  : HAS_CUML,
}

with open(f"{MODEL_DIR}/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved       → {MODEL_DIR}/metadata.json")

print("\n" + "=" * 50)
print("  All artifacts saved successfully.")
print("=" * 50)

In [ ]:
# ── 18. Model Reload Verification ─────────────────────────────────────────────
from pyspark.ml.classification import RandomForestClassificationModel

reloaded       = RandomForestClassificationModel.load(f"{MODEL_DIR}/spark_model")
reload_preds   = reloaded.transform(test_df.limit(1_000))
reload_metrics = spark_metrics(reload_preds)

print("Reloaded model — sanity check on 1,000 rows:")
print(f"  Accuracy : {reload_metrics['accuracy']:.4f}")
print(f"  F1       : {reload_metrics['f1']:.4f}")
print("Model reload OK ✅")

In [ ]:
# ── 19. Final Summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FINAL TRAINING SUMMARY")
print("=" * 60)
print(f"  Primary backend    : CPU (PySpark MLlib — distributed)")
if HAS_CUML:
    print(f"  GPU backend        : cuML RAPIDS — {GPU_INFO['gpus'][0]['name']}")
else:
    print(f"  GPU backend        : Not available")
print(f"  Train rows         : {n_train:>12,}")
print(f"  Test  rows         : {n_test:>12,}")
print(f"  Classes            : {NUM_CLASSES}  (Severity 1–4)")
print(f"  CV folds           : 3")
print(f"  Grid configs       : {len(param_grid)}")
print(f"  Best numTrees      : {best_model.getNumTrees}")
print(f"  Best maxDepth      : {best_model.getMaxDepth()}")
print(f"  Best minInstances  : {best_model.getMinInstancesPerNode()}")
print(f"  Best CV F1         : {cv_model.avgMetrics[best_idx]:.4f}")
print("-" * 60)
print(f"  Test Accuracy      : {metrics['accuracy']:.4f}")
print(f"  Test F1            : {metrics['f1']:.4f}")
print(f"  Test Precision     : {metrics['weightedPrecision']:.4f}")
print(f"  Test Recall        : {metrics['weightedRecall']:.4f}")
print("=" * 60)
print("  Saved artifacts:")
print(f"    {MODEL_DIR}/spark_model")
if HAS_CUML:
    print(f"    {MODEL_DIR}/cuml_model.pkl")
print(f"    {MODEL_DIR}/feature_importance.csv")
print(f"    {MODEL_DIR}/metadata.json")
print("  Plots:")
print("    rf_confusion_matrix.png")
print("    rf_feature_importance.png")
print("    rf_calibration.png")
print("    rf_tree_depth_dist.png")
print("=" * 60)

In [ ]:
spark.stop()
print("Spark session stopped.")